In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from torch_geometric.nn import GINConv, global_mean_pool, BatchNorm
from torch.utils.data import Dataset, DataLoader, random_split

from torch_geometric.data import Data, Batch
from torch_geometric.utils.smiles import from_smiles
from torch_geometric.loader.dataloader import Collater

import pandas as pd
import numpy as np
import os
import random
from datetime import datetime
from tqdm import tqdm
import joblib 

from src.configs.experiments_configs import cl_config

import faiss
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"



In [13]:
class TextEncoder(nn.Module):
    def __init__(self, model_name: str, pooling: str = "cls", device: str = "cuda"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.pooling = pooling
        self.device = device
        self.to(device)

    def forward(self, texts):
        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        ).to(self.device)

        outputs = self.encoder(**inputs)
        hidden = outputs.last_hidden_state  

        if self.pooling == "cls":
            emb = hidden[:, 0, :] 

        emb = nn.functional.normalize(emb, p=2, dim=-1)
        return emb

class GINBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GINBlock, self).__init__()
        self.gin = GINConv(nn.Sequential(
                    nn.Linear(in_features=in_channels, out_features=out_channels), 
                    nn.LeakyReLU(), 
                    nn.Linear(in_features=out_channels,out_features=out_channels)
                    ))
        self.bn = BatchNorm(out_channels)

    def forward(self, x, edge_index):
        x = self.gin(x, edge_index)
        x = self.bn(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.leaky_relu(x)

        return x
    

class GINEncoder(nn.Module):
    def __init__(self, hidden_dims: list ):
        super(GINEncoder, self).__init__()

        self.gins = nn.ModuleList()
       
        for in_dim, out_dim in zip(hidden_dims[:-1], hidden_dims[1:]):
            gin_block = GINBlock(in_channels=in_dim, out_channels=out_dim)
            self.gins.append(gin_block)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for block in self.gins:
            x = block(x, edge_index)
            
        out = global_mean_pool(x, batch) 
        return out

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )

    def forward(self, x):
        x = self.net(x)
        return torch.nn.functional.normalize(x, p=2, dim=-1)

In [4]:
class ContrastiveModel(nn.Module):
    def __init__(self, text_encoder, graph_encoder, text_dim=768, graph_dim=256, proj_dim=256):
        super().__init__()
        self.text_encoder = text_encoder
        self.graph_encoder = graph_encoder

        self.text_proj = ProjectionHead(text_dim, proj_dim)
        self.graph_proj = ProjectionHead(graph_dim, proj_dim)

    def forward(self, batch_graphs, passages):
        g_emb = self.graph_encoder(batch_graphs)             
        g_emb = self.graph_proj(g_emb)                      

        t_emb = self.text_encoder(passages)                   
        t_emb = self.text_proj(t_emb)                         

        return g_emb, t_emb

In [15]:
text_encoder = TextEncoder(cl_config.TEXT_ENCODER, pooling="cls", device=cl_config.DEVICE)
graph_encoder = GINEncoder(hidden_dims=cl_config.GNN_HIDDEN_DIMS).to(cl_config.DEVICE)

Some weights of BertModel were not initialized from the model checkpoint at bitshott/scibert_scivocab_chembl_passages_v1 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [30]:
model = ContrastiveModel(text_encoder, graph_encoder)
model.load_state_dict(torch.load("src/models/clmodel_dataset_300000_2025-09-26 17:21:11.451244_epoch_4.pt"))
model.eval()

ContrastiveModel(
  (text_encoder): TextEncoder(
    (encoder): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(31090, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, b

In [23]:
df = pd.read_csv('src/data/chembl_data_subsample.csv')
df = df.sample(n=300000, random_state=cl_config.RANDOM_SEED)


In [ ]:
graph_data = [from_smiles(smiles) for smiles in df['smiles']]   

In [25]:
passages = df['passage'].to_list()

In [46]:
mol_ids_list = df['mol_id'].to_list()

In [38]:
paired_embeddings = []

In [39]:
model.to(cl_config.DEVICE)

ContrastiveModel(
  (text_encoder): TextEncoder(
    (encoder): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(31090, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, b

In [40]:
with torch.no_grad():
    for graph, passage in tqdm(zip(graph_data, passages), total=len(passages)):
        graph.to(cl_config.DEVICE)
        g_emb = model.graph_proj(model.graph_encoder(graph))    
        t_emb = model.text_proj(model.text_encoder(passage))
        paired_embeddings.append((g_emb, t_emb))


100%|██████████| 300000/300000 [39:07<00:00, 127.77it/s] 


In [48]:
import faiss
import numpy as np

# Собираем графовые эмбеддинги
mol_ids = []
mol_embs = []

with torch.no_grad():
    for graph, mol_id in tqdm(zip(graph_data, mol_ids_list), total=len(mol_ids_list)):
        graph = graph.to(cl_config.DEVICE)
        g_emb = model.graph_proj(model.graph_encoder(graph))
        g_emb = g_emb.cpu().numpy()
        mol_embs.append(g_emb)
        mol_ids.append(mol_id)

mol_embs = np.vstack(mol_embs).astype("float32")

100%|██████████| 300000/300000 [07:23<00:00, 676.25it/s]


In [49]:
dim = mol_embs.shape[1]

# Косинусная близость через dot product после нормализации
faiss.normalize_L2(mol_embs)

index_mol = faiss.IndexFlatIP(dim)  # Inner Product
index_mol.add(mol_embs)

print("✅ Индекс готов. Всего молекул:", index_mol.ntotal)

✅ Индекс готов. Всего молекул: 300000


In [51]:
def search_molecules(query_text, top_k=5):
    # Текст в эмбеддинг
    with torch.no_grad():
        t_emb = model.text_proj(model.text_encoder(query_text))
        t_emb = t_emb.cpu().numpy().astype("float32")
        faiss.normalize_L2(t_emb)

    # Поиск по индексу
    D, I = index_mol.search(t_emb, top_k)

    results = []
    for idx, score in zip(I[0], D[0]):
        results.append((mol_ids[idx], float(score)))
    return results


# 🔎 Пример
hits = search_molecules("inhibition of DNA topoisomerase II", top_k=10)
for mol_id, score in hits:
    print(mol_id, score)

CHEMBL1214293 0.6761134266853333
CHEMBL4127712 0.6690033674240112
CHEMBL3640005 0.6629231572151184
CHEMBL1442698 0.6612467169761658
CHEMBL3581860 0.6548622846603394
CHEMBL3581860 0.6548622846603394
CHEMBL3942699 0.6453558206558228
CHEMBL1087710 0.6453535556793213
CHEMBL3938321 0.6438499689102173
CHEMBL1214419 0.6398996114730835


In [53]:
import torch
import faiss
import numpy as np
from tqdm import tqdm

mol_ids, texts = [], []
mol_embs, txt_embs = [], []

with torch.no_grad():
    for graph, passage, mol_id in tqdm(zip(graph_data, passages, mol_ids_list), total=len(passages)):
        # Граф → эмбеддинг
        graph = graph.to(cl_config.DEVICE)
        g_emb = model.graph_proj(model.graph_encoder(graph))
        g_emb = g_emb.cpu().numpy()

        # Текст → эмбеддинг
        t_emb = model.text_proj(model.text_encoder(passage))
        t_emb = t_emb.cpu().numpy()

        mol_embs.append(g_emb)
        txt_embs.append(t_emb)
        mol_ids.append(mol_id)
        texts.append(passage)

mol_embs = np.vstack(mol_embs).astype("float32")
txt_embs = np.vstack(txt_embs).astype("float32")

100%|██████████| 300000/300000 [39:38<00:00, 126.12it/s] 


In [54]:
dim = mol_embs.shape[1]

# Нормализация для cosine similarity через inner product
faiss.normalize_L2(mol_embs)
faiss.normalize_L2(txt_embs)

# Индексы
index_mol = faiss.IndexFlatIP(dim)   # Molecule space
index_txt = faiss.IndexFlatIP(dim)   # Text space

index_mol.add(mol_embs)
index_txt.add(txt_embs)

print("✅ IndexMol:", index_mol.ntotal, "✅ IndexTxt:", index_txt.ntotal)

✅ IndexMol: 300000 ✅ IndexTxt: 300000


In [55]:
def search_molecules(query_text, top_k=5):
    with torch.no_grad():
        q_emb = model.text_proj(model.text_encoder(query_text))
        q_emb = q_emb.cpu().numpy().astype("float32")
        faiss.normalize_L2(q_emb)

    D, I = index_mol.search(q_emb, top_k)
    return [(mol_ids[idx], float(score)) for idx, score in zip(I[0], D[0])]


hits = search_molecules("inhibition of DNA topoisomerase II", top_k=5)
for mol_id, score in hits:
    print("Mol:", mol_id, "Score:", score)

Mol: CHEMBL1214293 Score: 0.6761133670806885
Mol: CHEMBL4127712 Score: 0.6690033674240112
Mol: CHEMBL3640005 Score: 0.6629231572151184
Mol: CHEMBL1442698 Score: 0.6612467169761658
Mol: CHEMBL3581860 Score: 0.6548622846603394


In [56]:
def search_texts(graph, top_k=5):
    with torch.no_grad():
        g_emb = model.graph_proj(model.graph_encoder(graph.to(cl_config.DEVICE)))
        g_emb = g_emb.cpu().numpy().astype("float32")
        faiss.normalize_L2(g_emb)

    D, I = index_txt.search(g_emb, top_k)
    return [(texts[idx], float(score)) for idx, score in zip(I[0], D[0])]


hits = search_texts(graph_data[0], top_k=5)
for passage, score in hits:
    print("Text:", passage, "Score:", score)

Text: Compound CHEMBL3220950 > IC50 = 10000.0 nM against N-arachidonyl glycine receptor (Homo sapiens) in assay: Antagonist activity against human GPR18 expressed in CHO cells assessed as reduction in delta9-THC-induced beta-arrestin recruitment by beta-galactosidase enzyme fragment complementation method [Medchemcomm, 2014.0] Score: 0.5912903547286987
Text: Compound CHEMBL3221189 > Ki = 10000.0 nM against Cannabinoid CB1 receptor (Homo sapiens) in assay: Displacement of [3H]CP55,940 from human recombinant CB1 receptor expressed in CHO-K1 cells [Medchemcomm, 2014.0] Score: 0.5855593681335449
Text: Compound CHEMBL3221181 = Inhibition = 7.0 % against Cannabinoid CB1 receptor (Homo sapiens) in assay: Displacement of [3H]CP55,940 from human recombinant CB1 receptor expressed in CHO-K1 cells at 10 uM [Medchemcomm, 2014.0] Score: 0.5843298435211182
Text: Compound CHEMBL3221187 = Ki = 1110.0 nM against Cannabinoid CB2 receptor (Homo sapiens) in assay: Displacement of [3H]CP55,940 from human r

In [ ]:
def evaluate_mol2text(model, index_txt, passages, mol_ids, graph_data, top_k=10):
    hits = 0
    rr_sum = 0.0
    ndcg_sum = 0.0
    total = len(graph_data)

    for graph, true_passage in tqdm(zip(graph_data, passages), total=total, desc="Eval Mol→Text"):
        with torch.no_grad():
            g_emb = model.graph_proj(model.graph_encoder(graph.to(cl_config.DEVICE)))
            g_emb = g_emb.cpu().numpy().astype("float32")
            faiss.normalize_L2(g_emb)

        D, I = index_txt.search(g_emb, top_k)
        retrieved_passages = [passages[idx] for idx in I[0]]

        if true_passage in retrieved_passages:
            hits += 1
            rank = retrieved_passages.index(true_passage) + 1
            rr_sum += 1.0 / rank
            ndcg_sum += 1.0 / np.log2(rank + 1)

    recall_at_k = hits / total
    mrr_at_k = rr_sum / total
    ndcg_at_k = ndcg_sum / total

    return {"Recall@K": recall_at_k, "MRR@K": mrr_at_k, "NDCG@K": ndcg_at_k}


# Пример
metrics = evaluate_mol2text(model, index_txt, passages, mol_ids, graph_data, top_k=10)
print(metrics)

Eval Mol→Text:   1%|          | 2644/300000 [01:11<2:14:55, 36.73it/s]

In [ ]:
import numpy as np
from tqdm import tqdm

def evaluate_text2mol(model, index_mol, passages, mol_ids, batch_graphs, top_k=10):
    hits = 0
    rr_sum = 0.0
    ndcg_sum = 0.0
    total = len(passages)

    for passage, true_mol_id in tqdm(zip(passages, mol_ids), total=total, desc="Eval Text→Mol"):
        with torch.no_grad():
            q_emb = model.text_proj(model.text_encoder(passage))
            q_emb = q_emb.cpu().numpy().astype("float32")
            faiss.normalize_L2(q_emb)

        D, I = index_mol.search(q_emb, top_k)

        retrieved_ids = [mol_ids[idx] for idx in I[0]]

        # Recall@K
        if true_mol_id in retrieved_ids:
            hits += 1

            # Rank
            rank = retrieved_ids.index(true_mol_id) + 1
            rr_sum += 1.0 / rank
            ndcg_sum += 1.0 / np.log2(rank + 1)

    recall_at_k = hits / total
    mrr_at_k = rr_sum / total
    ndcg_at_k = ndcg_sum / total

    return {"Recall@K": recall_at_k, "MRR@K": mrr_at_k, "NDCG@K": ndcg_at_k}


# Пример
metrics = evaluate_text2mol(model, index_mol, passages, mol_ids, graph_data, top_k=10)
print(metrics)

Eval Text→Mol:   2%|▏         | 5563/300000 [02:56<2:35:37, 31.53it/s]


KeyboardInterrupt: 